In [ ]:
import sys
sys.path.insert(0, '../backend/src')

import networkx as nx
from collections import Counter
from core.dataset_manager import get_primekg_drug_repurposing
from fol.learning.learner import ExplanationLearner

G = get_primekg_drug_repurposing()
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

nt = Counter(d.get('node_type') for _, d in G.nodes(data=True))
et = Counter(d.get('edge_type') for _, _, d in G.edges(data=True))
print(f"\nNode types: {dict(sorted(nt.items()))}")
print(f"Edge types: {dict(sorted(et.items()))}")

# Key subsets
drug_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'drug']
gene_types = {'bridge_gene', 'drug_target', 'disease_gene'}
gene_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') in gene_types]
disease_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'disease']

print(f"\n{len(drug_nodes)} drugs, {len(gene_nodes)} genes "
      f"(bridge={nt.get('bridge_gene',0)}, target={nt.get('drug_target',0)}, "
      f"disease={nt.get('disease_gene',0)}), {len(disease_nodes)} diseases")

# Drug subgraph for Task 1 (prevents node_type as trivial separator)
G_drugs = G.subgraph(drug_nodes).copy()

# Sample drug attributes
sample = next(n for n in drug_nodes if G.nodes[n].get('categories'))
print(f"\nSample drug ({G.nodes[sample].get('display_name')}):")
for k, v in G.nodes[sample].items():
    if k not in ('node_type',):
        val = v[:3] if isinstance(v, list) and len(v) > 3 else v
        print(f"  {k}: {val}")

## Task 1: Conjunctive — CNS Drug Pharmacological Fingerprint

**Hypothesis.** A pharmaceutical researcher selects all drugs indicated for major neuropsychiatric conditions (depression, anxiety, schizophrenia, Parkinson's, migraine) and asks: *what pharmacological and molecular features characterise CNS-active drugs relative to the drug corpus?* The analyst does not know in advance which ATC classifications, drug categories, or molecular properties will emerge — the system must discover the fingerprint from the selection alone.

**Selection.** Drugs connected via `indication` edges to neuropsychiatric diseases, projected onto the drug subgraph ($N = 929$ drugs). The drug subgraph contains no edges (drugs connect to genes and diseases, not to each other), forcing the learner to rely on drug attributes: pharmacological categories, ATC classifications, molecular weight, TPSA, cLogP, drug interaction count.

**Learning mode.** Conjunctive (beam search, default hyperparameters).

In [ ]:
# Select drugs indicated for neuropsychiatric diseases
neuro_keywords = {'schizophrenia', 'depress', 'anxiety', 'parkinson', 'migraine'}
neuro_diseases = {
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'disease'
    and any(kw in d.get('display_name', '').lower() for kw in neuro_keywords)
}
print(f"Neuropsychiatric diseases matched: {len(neuro_diseases)}")
for nd in sorted(neuro_diseases):
    print(f"  {G.nodes[nd].get('display_name')}")

cns_drugs = list({
    nb for disease in neuro_diseases for nb in G.neighbors(disease)
    if G.nodes[nb].get('node_type') == 'drug'
})
pi = len(cns_drugs) / len(drug_nodes)
print(f"\nS⁺: {len(cns_drugs)} CNS drugs  vs  {len(drug_nodes)} total drugs  (π = {pi:.3f})")

# Learn
result = ExplanationLearner(beam_width=8, top_k=5).learn_explanations(G_drugs, cns_drugs)
print(f"\n{'─'*90}")
print(f"Learning time: {result.learning_time_ms:.0f} ms")
for i, c in enumerate(result.clauses, 1):
    pi_c = c.p / (c.p + c.n) if (c.p + c.n) > 0 else 0
    enrich = pi_c / pi
    print(f"\n  C{i}  score={c.score:.1f}  coverage={c.coverage:.0%}  "
          f"precision={pi_c:.0%}  enrichment={enrich:.1f}×")
    print(f"      {c.fol_expression}")
    print(f"      p={c.p} CNS drugs, n={c.n} non-CNS drugs")

### Insight

The learner surfaces a layered pharmacological fingerprint of CNS drugs:

1. **ATC nervous system classification** — `atc_1(x) = "nervous system"` covers 58% of the selection at 87% precision (3.3× enrichment). The system discovers the WHO classification link without being told to look for it.
2. **Antidepressant + nervous system conjunction** — refining with `categories(x) = "Antidepressive Agents"` pushes precision to 99% for 35% of the selection (3.8× enrichment), isolating the largest pharmacological subclass.
3. **Antidepressive agents alone** — `categories(x) = "Antidepressive Agents"` at 91% precision and 40% coverage demonstrates that antidepressants are the dominant drug class in the CNS selection.
4. **CNS Depressants** — `categories(x) = "Central Nervous System Depressants"` achieves 74% precision and 59% coverage, capturing the second major pharmacological class (anxiolytics, sedatives, anticonvulsants).
5. **High-interaction CNS depressants** — `atc_1(x) = "nervous system" ∧ categories(x) = "Central Nervous System Depressants" ∧ drug_interaction_count(x) ≥ 3256.0` achieves 100% precision by adding a numeric threshold on drug interactions. CNS drugs that are also depressants AND highly interactive (≥3,256 known drug-drug interactions) are *exclusively* neuropsychiatric.

**Practical impact.** The fingerprint reveals that CNS drugs are characterised by a two-axis signature: pharmacological class (antidepressants + CNS depressants) and interaction burden. The drug interaction threshold is a non-obvious finding — it implies that high polypharmacy risk is a hallmark of this therapeutic area. A screening pipeline can use these predicates to flag CNS-like molecular profiles in novel compounds.

## Task 2: Contrastive — Cancer-Exclusive vs Autoimmune-Exclusive Gene Targets

**Hypothesis.** A genomics researcher asks: *what distinguishes gene targets implicated exclusively in cancer from those implicated exclusively in autoimmune/inflammatory disease?* While both categories involve immune dysregulation, the underlying mechanisms differ. The contrastive analysis should surface biological and structural differences in the gene-disease network.

**Selection.** S⁺ = genes connected to cancer diseases but not autoimmune diseases; S⁻ = genes connected to autoimmune diseases but not cancer. Overlap excluded. Full graph used (typed 2-hop paths available).

**Learning mode.** Contrastive (restricted universe: S⁺ ∪ S⁻).

In [ ]:
# Build cancer and autoimmune disease sets
cancer_keywords = {'cancer', 'carcinoma', 'leukemia', 'lymphoma', 'neoplasm', 'sarcoma'}
autoimmune_keywords = {'arthritis', 'asthma', 'rhinitis', 'eczema', 'spondyloarthropathy', 'dermatitis'}

cancer_diseases = {
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'disease'
    and any(kw in d.get('display_name', '').lower() for kw in cancer_keywords)
}
autoimmune_diseases = {
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'disease'
    and any(kw in d.get('display_name', '').lower() for kw in autoimmune_keywords)
}
print(f"Cancer diseases: {len(cancer_diseases)}")
print(f"Autoimmune diseases: {len(autoimmune_diseases)}")

# Gene targets connected to each disease group
cancer_genes = {
    nb for d in cancer_diseases for nb in G.neighbors(d)
    if G.nodes[nb].get('node_type') in gene_types
}
autoimmune_genes = {
    nb for d in autoimmune_diseases for nb in G.neighbors(d)
    if G.nodes[nb].get('node_type') in gene_types
}

# Exclusive sets
s_plus = list(cancer_genes - autoimmune_genes)
s_minus = list(autoimmune_genes - cancer_genes)
overlap = cancer_genes & autoimmune_genes
pi_c = len(s_plus) / (len(s_plus) + len(s_minus))
print(f"\nCancer-exclusive genes (S⁺): {len(s_plus)}")
print(f"Autoimmune-exclusive genes (S⁻): {len(s_minus)}")
print(f"Overlap (excluded): {len(overlap)}")
print(f"π = {pi_c:.3f}")

# Gene subtype breakdown
for label, genes in [("Cancer-exclusive", s_plus), ("Autoimmune-exclusive", s_minus)]:
    subtypes = Counter(G.nodes[g].get('node_type') for g in genes)
    print(f"\n{label} gene subtypes: {dict(subtypes)}")

# Contrastive learning
result = ExplanationLearner(beam_width=8, top_k=5).learn_explanations(
    G, s_plus, contrast_nodes=s_minus
)
print(f"\n{'─'*90}")
print(f"Learning time: {result.learning_time_ms:.0f} ms")
for i, c in enumerate(result.clauses, 1):
    pi_c_clause = c.p / (c.p + c.n) if (c.p + c.n) > 0 else 0
    enrich = pi_c_clause / pi_c
    print(f"\n  C{i}  score={c.score:.1f}  coverage={c.coverage:.0%}  "
          f"precision={pi_c_clause:.0%}  enrichment={enrich:.1f}×")
    print(f"      {c.fol_expression}")
    print(f"      p={c.p} cancer-exclusive, n={c.n} autoimmune-exclusive")

### Insight

The contrastive analysis surfaces both biological and structural differences between cancer and autoimmune gene targets:

1. **Typed 2-hop path structure** — the top clauses use universal quantifiers over drug-protein and disease-protein typed paths. These capture local network topology: cancer-exclusive genes sit in neighborhoods where *all* reachable nodes via contraindication and disease-disease paths are disease genes — meaning cancer genes lack the drug-target connectivity that autoimmune genes have. The autoimmune-exclusive genes are more directly druggable.
2. **Nucleoplasm localisation** — `cellular_components(x) = "nucleoplasm" ∧ node_type(x) = "disease_gene"` at 98% precision identifies nuclear-localized disease genes as a cancer hallmark. This is biologically meaningful: cancer-exclusive genes are predominantly nuclear proteins involved in gene regulation and transcription — consistent with the known role of transcription factors and chromatin remodelers in oncogenesis.
3. **Structural community** — `louvain_community(x) = "cluster_1"` at 100% precision reveals that cancer-exclusive genes form a distinct structural community in the knowledge graph, implying tighter interconnection among themselves than with autoimmune-related genes.

**Practical impact.** The contrastive predicates reveal that cancer-exclusive genes are structurally more isolated from drug space (they are disease genes, not bridge genes or drug targets) and biologically concentrated in the nucleus. This suggests that targeting these genes directly is more challenging — repurposing strategies may need to target their pathway neighbours rather than the genes themselves. The community structure finding could guide network-based prioritisation of indirect therapeutic targets.

## Task 3: Path-Derived Selection — Schizophrenia–Parkinson Shared Genetic Bridges

**Hypothesis.** Schizophrenia and Parkinson's disease are distinct neurological conditions, yet clinical observations suggest shared biological mechanisms (particularly involving dopaminergic pathways). The analyst computes 3-hop paths between the two disease nodes and asks: *what characterises the gene hubs that structurally connect schizophrenia to Parkinson's disease?* These bridging genes represent shared molecular mechanisms and potential multi-indication drug targets.

**Selection.** All intermediate gene nodes on 3-hop paths between schizophrenia and Parkinson's disease in the full graph. The full graph is used for learning, enabling typed 2-hop path literals.

**Learning mode.** Conjunctive (full graph with cross-type neighbourhood reasoning).

In [ ]:
# Schizophrenia ↔ Parkinson's disease: 3-hop paths
schizo = '28313'
parkinson = '28547'
print(f"Source: {G.nodes[schizo].get('display_name')}")
print(f"Target: {G.nodes[parkinson].get('display_name')}")

# Compute all 3-hop paths
path_nodes = set()
for path in nx.all_simple_paths(G, schizo, parkinson, cutoff=3):
    path_nodes.update(path[1:-1])

int_types = Counter(G.nodes[n].get('node_type') for n in path_nodes)
print(f"\n3-hop intermediates: {len(path_nodes)} nodes")
print(f"  Types: {dict(int_types)}")

# Restrict to gene nodes for learning
path_genes = [n for n in path_nodes if G.nodes[n].get('node_type') in gene_types]
pi = len(path_genes) / G.number_of_nodes()
print(f"\nS⁺: {len(path_genes)} bridging genes  (π = {pi:.3f})")

# Gene subtype breakdown
gene_subtypes = Counter(G.nodes[n].get('node_type') for n in path_genes)
print(f"  Gene subtypes: {dict(gene_subtypes)}")

# How many are bridge genes (druggable)?
bridge_in_path = sum(1 for n in path_genes if G.nodes[n].get('node_type') == 'bridge_gene')
print(f"  Of which {bridge_in_path} are bridge genes (drug target ∩ disease gene)")

# Learn
result = ExplanationLearner(beam_width=8, top_k=5).learn_explanations(G, path_genes)
print(f"\n{'─'*90}")
print(f"Learning time: {result.learning_time_ms:.0f} ms")
for i, c in enumerate(result.clauses, 1):
    pi_c = c.p / (c.p + c.n) if (c.p + c.n) > 0 else 0
    enrich = pi_c / pi
    print(f"\n  C{i}  score={c.score:.1f}  coverage={c.coverage:.0%}  "
          f"precision={pi_c:.0%}  enrichment={enrich:.1f}×")
    print(f"      {c.fol_expression}")
    print(f"      p={c.p} bridging genes, n={c.n} non-bridging")

### Insight

The 73 bridging genes between schizophrenia and Parkinson's disease are characterised by dense multi-scale connectivity:

1. **Drug indication + contraindication hub structure** — the top clause combines `at_least(2)` drug-contraindication 2-hop paths with `at_least(3)` drug-indication 2-hop paths: genes on the schizophrenia–Parkinson bridge connect to many drugs via both indication and contraindication edges. This reveals pharmacological duality: the bridging genes are targeted by drugs that treat multiple conditions but also carry safety concerns across conditions — the hallmark of polypharmacy risk in neuropsychiatric treatment.
2. **Off-label use pathway** — `∃y ∈ N_{drug_protein.off-label use}(x) : degree(y) ≥ 20.5` at 10.3× enrichment and 49% precision. Bridging genes connect to drugs that are already used off-label for conditions beyond their primary indication — direct evidence that these genes sit at the intersection of existing repurposing activity.
3. **Disease-indication connectivity** — deep neighbourhood predicates combining disease-protein and drug-protein paths at high enrichments (6.6–9.1×). The bridging genes are not isolated proteins but hubs in the drug-disease network, connected to both disease-associated and drug-targeting pathways.

**Practical impact.** The bridging genes represent shared molecular mechanisms between two major neuropsychiatric conditions. The off-label use signal is particularly actionable: drugs already used off-label via these genes are candidates for formal clinical evaluation in the other disease. The pharmacological duality (indication + contraindication) warns that repurposing requires careful safety profiling — the same mechanisms that make these genes therapeutic targets also make them liability risks across conditions.

## Summary

| Task | Mode | Selection | π | Top enrichment | Key finding |
|---|---|---|---|---|---|
| 1. CNS Fingerprint | Conjunctive | 243 neuro drugs | 0.262 | 3.8× | ATC nervous system + antidepressant + drug interaction threshold |
| 2. Cancer vs Autoimmune | Contrastive | 113 vs 46 genes | 0.711 | 1.4× | Nucleoplasm localisation, reduced druggability, community structure |
| 3. Schizo↔Parkinson | Path-derived | 73 bridging genes | 0.047 | 10.3× | Off-label use pathways, indication/contraindication hub duality |

Three tasks demonstrate cross-space reasoning in biomedical knowledge graphs — from molecular fingerprinting (drug attributes) through disease mechanism comparison (typed 2-hop paths) to cross-disease drug repurposing (path-derived selections).